# Grounded RAG Generation Workflow

Run the cells from top to bottom. The notebook parses documents, prepares retrieval, inspects evidence, renders a grounded prompt, and generates an answer with sources.

In [1]:
import sys
from pathlib import Path

from chromadb.api.client import SharedSystemClient

# Reset process-local clients when a persistent index was removed or rebuilt.
SharedSystemClient.clear_system_cache()

project_root = Path.cwd()
if not (project_root / "src").exists():
    for parent in project_root.resolve().parents:
        if (parent / "src").exists():
            project_root = parent
            break

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.core.config import Config
from src.core.logger import setup_logging
from src.document.chunker import chunk_sections
from src.document.pdf_parser import parse_pdf_document
from src.document.word_parser import parse_word_document
from src.generation.llm import LLM
from src.generation.prompts import PromptBuilder
from src.generation.rag_pipeline import RAGPipeline
from src.generation.response import RAGResponse, SourceReference
from src.retrieval.bm25_retriever import BM25Retriever
from src.retrieval.embedder import Embedder
from src.retrieval.hybrid_retriever import HybridRetriever
from src.retrieval.reranker import Reranker
from src.retrieval.vector_store import VectorStore

config = Config()
setup_logging(config)
print(f"Project root: {project_root}")

2026-08-10 13:31:31,616 | INFO | src | Logging configured with level INFO


Project root: /Users/humengqing/Documents/Code/VSCode/doc-qa-agent


## 1. Parse documents and create chunks

Configured PDF and DOCX files are parsed into one section list, then converted to chunks with stable IDs and source metadata.

In [2]:
configured_paths = config.get("documents", "paths", default=[])
if not configured_paths:
    raise ValueError("documents.paths must contain at least one document")

sections = []
for configured_path in configured_paths:
    document_path = Path(configured_path)
    if not document_path.is_absolute():
        document_path = project_root / document_path

    # Select the parser from the document type to preserve one shared schema.
    if document_path.suffix.lower() == ".pdf":
        sections.extend(parse_pdf_document(document_path, config))
    elif document_path.suffix.lower() == ".docx":
        sections.extend(parse_word_document(document_path))
    else:
        raise ValueError(f"Unsupported document type: {document_path}")

chunks = chunk_sections(sections, config)
print(f"Parsed {len(sections)} section(s) into {len(chunks)} chunk(s)")
print(chunks[0]["chunk_id"])
print(chunks[0]["metadata"])

2026-08-10 13:31:34,333 | INFO | src.document.pdf_parser | Parsed HuMengqing.pdf into 61 section(s)
2026-08-10 13:31:34,582 | INFO | src.document.word_parser | Parsed Report.docx into 62 section(s)
2026-08-10 13:31:34,585 | INFO | src.document.chunker | Created 187 chunk(s) from 123 section(s)


Parsed 123 section(s) into 187 chunk(s)
HuMengqing_chunk_001
{'source': 'HuMengqing.pdf', 'page': -9, 'section_title': 'Learning on OCT-data', 'chunk_type': 'text', 'chunk_index': 1}


## 2. Build dense and sparse retrieval

Chunks are embedded once for ChromaDB and tokenized for the process-local BM25 index. Both retrieval methods use the same chunk IDs and text.

In [3]:
# Reuse the computed vectors during Chroma upsert to avoid duplicate encoding.
embedder = Embedder(config)
chunk_embeddings = embedder.embed_chunks(chunks)
for chunk, embedding in zip(chunks, chunk_embeddings):
    chunk["embedding"] = embedding

vector_store = VectorStore(config, embedder=embedder)
vector_store.add_chunks(chunks)

# BM25 is intentionally rebuilt because its index is not persisted.
bm25_retriever = BM25Retriever(chunks, config)
print(f"Stored chunks: {vector_store.collection.count()}")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

2026-08-10 13:31:45,463 | INFO | src.retrieval.embedder | Loaded embedding model: BAAI/bge-large-en-v1.5
2026-08-10 13:32:06,718 | INFO | src.retrieval.vector_store | Upserted 187 chunk(s) into collection doc_chunks
2026-08-10 13:32:06,735 | INFO | src.retrieval.bm25_retriever | Built BM25 index for 187 chunk(s)


Stored chunks: 187


## 3. Assemble the pipeline

The individual components are retained as variables for inspection. `RAGPipeline` is the application-facing composition of the same components.

In [4]:
hybrid_retriever = HybridRetriever(vector_store, bm25_retriever, config)
reranker = Reranker(config)
prompt_builder = PromptBuilder(config)
llm = LLM(config)

pipeline = RAGPipeline(
    hybrid_retriever,
    reranker,
    prompt_builder,
    llm,
)
print("RAG pipeline is ready")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

2026-08-10 13:32:12,327 | INFO | src.retrieval.reranker | Loaded reranking model: cross-encoder/ms-marco-MiniLM-L-6-v2


RAG pipeline is ready


## 4. Inspect final evidence before calling the LLM

This step performs dense plus BM25 retrieval, RRF fusion, and Cross-Encoder reranking. It does not make a remote LLM request.

In [5]:
query = "What is the classification accuracy of ResNet26-V2?"

# RRF combines dense and keyword candidates into one ranked list.
hybrid_results = hybrid_retriever.search(query)

# Cross-Encoder jointly scores the query and each candidate for final ranking.
reranked_results = reranker.rerank(query, hybrid_results)

print(f"Hybrid candidates: {len(hybrid_results)}")
print(f"Final evidence chunks: {len(reranked_results)}")
for rank, chunk in enumerate(reranked_results, start=1):
    print(f"\n{rank}. {chunk['chunk_id']} | Rerank: {chunk['rerank_score']:.6f}")
    print(chunk["metadata"])
    print(chunk["text"][:300])

2026-08-10 13:32:13,909 | INFO | src.retrieval.hybrid_retriever | Fused 20 dense and 20 BM25 result(s) into 10 chunk(s)
2026-08-10 13:32:14,316 | INFO | src.retrieval.reranker | Reranked 10 candidate(s) and returned 5 result(s)


Hybrid candidates: 10
Final evidence chunks: 5

1. HuMengqing_chunk_072 | Rerank: 7.255227
{'chunk_index': 72, 'source': 'HuMengqing.pdf', 'page': 52, 'chunk_type': 'text', 'section_title': '4.4 Comparison with Other Convolutional Neural Networks'}
study has about 3.5 million parameters, whereas EfficientNet-B0, which is used for binary
classification tasks, has about 5.3 million parameters and VGG16 has 138 million
parameters. It is worth noting that ResNet26-V2 achieves better performance with far
fewer parameters than the other models, indi

2. HuMengqing_chunk_075 | Rerank: 5.132246
{'chunk_index': 75, 'source': 'HuMengqing.pdf', 'page': 54, 'section_title': '5 Summary', 'chunk_type': 'text'}
the infrastructure in this study, and the BottleneckV2 module is adopted for the design of
the network. To optimize the model performance, the width multiplication factor (K) and
the number of bottleneck modules (N) of the network are adjusted, respectively. After
several rounds of experiments

## 5. Render the grounded prompt

Inspect this output before generation when validating retrieval quality or modifying the Prompt template. It contains actual chunk IDs and metadata for traceability.

In [6]:
prompt = prompt_builder.build(query, reranked_results)
print(prompt)

# Role

You are a research document question-answering assistant. Answer the user's question only from the provided context.

# Rules

- Do not add information that is absent from the context.
- If the context does not contain enough information, say: "Based on the available documents, I cannot answer this question."
- Keep numerical values exactly as they appear in the context.
- End the answer with the chunk ID or IDs that support it.

# Context

[Chunk ID: HuMengqing_chunk_072]
Source: HuMengqing.pdf | Page: 52 | Section: 4.4 Comparison with Other Convolutional Neural Networks
Content:
study has about 3.5 million parameters, whereas EfficientNet-B0, which is used for binary
classification tasks, has about 5.3 million parameters and VGG16 has 138 million
parameters. It is worth noting that ResNet26-V2 achieves better performance with far
fewer parameters than the other models, indicating that it significantly improves the
computational efficiency and memory footprint while maintainin

## 6. Generate an answer and display sources

This cell sends one request to ScaDS.AI. The source list is constructed from the reranked evidence rather than inferred from the generated answer.

In [7]:
# This request uses SCADS_API_KEY from .env. Do not print the key.
answer = llm.generate(prompt)
sources = tuple(
    SourceReference(
        chunk_id=chunk["chunk_id"],
        source=chunk["metadata"].get("source", "unknown source"),
        page=chunk["metadata"].get("page"),
        section_title=chunk["metadata"].get("section_title"),
    )
    for chunk in reranked_results
)
response = RAGResponse(answer=answer, sources=sources)

print(response.answer)
print("\nSources:")
for source in response.sources:
    location = f"page {source.page}" if source.page is not None else "page unknown"
    section = source.section_title or "section unknown"
    print(f"- {source.chunk_id} | {source.source} | {location} | {section}")

2026-08-10 13:32:16,651 | INFO | src.generation.llm | Generated answer with 118 character(s)


The classification accuracy of ResNet26-V2 is 0.9434. HuMengqing_chunk_071, HuMengqing_chunk_075, HuMengqing_chunk_099

Sources:
- HuMengqing_chunk_072 | HuMengqing.pdf | page 52 | 4.4 Comparison with Other Convolutional Neural Networks
- HuMengqing_chunk_075 | HuMengqing.pdf | page 54 | 5 Summary
- HuMengqing_chunk_071 | HuMengqing.pdf | page 52 | 4.4 Comparison with Other Convolutional Neural Networks
- HuMengqing_chunk_099 | HuMengqing.pdf | page 53 | Table 8
- HuMengqing_chunk_054 | HuMengqing.pdf | page 35 | 3.3.1 ResNet Model Structure


## Application shortcut

In an application, replace the manual inspection cells with `pipeline.answer(query)`. Do not run the shortcut immediately after the generation cell because it repeats retrieval and sends another LLM request.

In [9]:
response = pipeline.answer(query)
print(response.answer)

2026-08-10 13:32:44,918 | INFO | src.retrieval.hybrid_retriever | Fused 20 dense and 20 BM25 result(s) into 10 chunk(s)
2026-08-10 13:32:45,846 | INFO | src.retrieval.reranker | Reranked 10 candidate(s) and returned 5 result(s)
2026-08-10 13:32:47,956 | INFO | src.generation.llm | Generated answer with 118 character(s)
2026-08-10 13:32:47,958 | INFO | src.generation.rag_pipeline | Answered question with 5 retrieved source(s)


The classification accuracy of ResNet26-V2 is 0.9434. HuMengqing_chunk_071, HuMengqing_chunk_075, HuMengqing_chunk_099
